# 🦜 LangChain + Groq — Beginner to Intermediate
### Hands-On Activity Guide

---

**What you'll learn in order:**

| Part | Topic | Level |
|---|---|---|
| 0 | Setup & first call | Beginner |
| 1 | Prompt Templates | Beginner |
| 2 | Chains (LCEL) | Beginner |
| 3 | Output Parsers | Beginner–Intermediate |
| 4 | Conversation Memory | Intermediate |
| 5 | Tools & Agents | Intermediate |
| 6 | RAG — Retrieval-Augmented Generation | Intermediate |
| 7 | Capstone Project | Intermediate |

> ⏱ **Time:** ~2 hours &nbsp;|&nbsp; 💰 **Cost:** Free (Groq free tier)  
> 📦 All concepts build on each other — do them in order.


---
## Part 0 — Setup


In [34]:
!pip install langchain langchain-groq langchain-community \
             langchain-core faiss-cpu sentence-transformers --quiet
print("✅ Done!")


✅ Done!


### Get your Groq API key
1. Sign up at [console.groq.com](https://console.groq.com)
2. Go to **API Keys → Create API Key**
3. Paste it below

> ⚠️ **Never push this notebook to GitHub with the key filled in.**  
> In real projects, load the key from a `.env` file using `python-dotenv`.


In [ ]:
import os
os.environ["GROQ_API_KEY"] = "gsk_..."   # ← paste your key here


In [9]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [11]:
from langchain_groq import ChatGroq

# This is the LangChain wrapper around Groq's API
llm = ChatGroq(
    model="llama-3.1-8b-instant",   # fast, free Llama 3 model
    temperature=0.7            # 0 = deterministic, 1 = creative
)

# Quick smoke-test
response = llm.invoke("Say hello in one sentence.")
print(response.content)


Hello, how can I assist you today?


### 🔍 What is LangChain?

LangChain is a **framework** that gives you reusable building blocks for LLM apps:

```
Prompt Template  →  LLM  →  Output Parser
      ↑                           ↓
   Variables                Structured data
```

Instead of writing raw API calls and string manipulation every time,  
LangChain lets you **compose** these pieces with a clean `|` pipe operator.

Think of it like UNIX pipes, but for AI.


---
## Part 1 — Prompt Templates

Hardcoding prompts is fragile. A **PromptTemplate** lets you define a prompt once with `{placeholders}` and fill them in at runtime — cleanly and reusably.


In [12]:
from langchain_core.prompts import ChatPromptTemplate

# Define a reusable template with {placeholders}
template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Reply in a {tone} tone."),
    ("human",  "{question}")
])

# Fill in the placeholders and call the model
chain = template | llm   # ← the pipe operator chains template → model

result = chain.invoke({
    "role":     "friendly science teacher",
    "tone":     "simple and enthusiastic",
    "question": "What is gravity?"
})
print(result.content)


Gravity is one of my favorite topics in science. So, you know how things fall down when you drop them, right? Like if you drop a ball or a pencil, it falls towards the ground. That's because of something called gravity!

Gravity is a force that pulls objects towards each other. The Earth has gravity, and it pulls everything towards it, including you, me, and even the air around us. That's why we don't float off into space when we're standing on the Earth.

Imagine the Earth as a big magnet. Just like how magnets attract certain metals, the Earth's gravity attracts everything towards it. But instead of being a magnetic force, it's a gravitational force that pulls things down.

Gravity is also what keeps us on the ground and what makes things fall. It's a really strong force, but it's also what helps us stay safe and secure on the Earth. Isn't that cool?


### What just happened?

1. `ChatPromptTemplate` defined the structure with `{placeholders}`
2. The `|` operator created a **chain**: template feeds into llm
3. `.invoke({...})` filled the variables and ran the whole thing

### 📝 Task 1
> Change `role` to `"strict university professor"` and `tone` to `"formal"`.  
> Ask the same question. Notice how the answer changes with no prompt rewriting.


In [13]:
# Task 1 — change the role and tone
result = chain.invoke({
    "role":     "strict university professor",
    "tone":     "formal",
    "question": "What is gravity?"
})
print(result.content)


A question that has puzzled scholars for centuries.  Gravity, in its most fundamental form, is a fundamental force of nature that governs the interaction between masses.  It is a universal force that attracts two objects with mass towards each other, with the strength of the force depending on the mass of the objects and the distance between them.

Historically, the concept of gravity has undergone significant development, from ancient philosophers like Aristotle to modern theories like Einstein's General Relativity.  While Aristotle believed that objects fell towards the ground due to a natural tendency, Sir Isaac Newton formulated the Law of Universal Gravitation in 1687, which states that every point mass attracts every other point mass by a force acting along the line intersecting both points.

Newton's law of gravity can be represented mathematically as:

F = G * (m1 * m2) / r^2

Where F is the gravitational force between two objects, G is the gravitational constant, m1 and m2 are

In [14]:
# Prompt templates also work with plain strings (not just chat format)
from langchain_core.prompts import PromptTemplate

summary_template = PromptTemplate.from_template(
    "Summarise the following text in {num} bullet points:\n\n{text}"
)

# Format it without calling the model — just to inspect
filled = summary_template.format(
    num=3,
    text="LangChain is a framework for building LLM-powered applications. "
         "It provides chains, memory, tools, and agents."
)
print("Formatted prompt:")
print(filled)


Formatted prompt:
Summarise the following text in 3 bullet points:

LangChain is a framework for building LLM-powered applications. It provides chains, memory, tools, and agents.


---
## Part 2 — Chains (LCEL)

LangChain Expression Language (**LCEL**) lets you build pipelines by chaining components with `|`.  
Each component's output becomes the next component's input.


### 2.1 — Simple Chain

```
PromptTemplate  →  ChatGroq  →  result
```


In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# StrOutputParser extracts just the text from the model's response object
parser = StrOutputParser()

explain_chain = (
    ChatPromptTemplate.from_template(
        "Explain {concept} to a {audience} in under 4 sentences."
    )
    | llm
    | parser    # ← turns AIMessage object into a plain string
)

result = explain_chain.invoke({
    "concept":  "neural networks",
    "audience": "10-year-old"
})
print(result)
print(type(result))   # <-- now a plain str, not an AIMessage object


Imagine you have a really good friend who can recognize your face. They've seen you a lot, so they know what your eyes, nose, and smile look like. A neural network is like a super smart computer program that can "learn" to recognize things, like pictures of cats or dogs, by looking at lots of examples and getting better and better at it.
<class 'langchain_core.messages.base.TextAccessor'>


### 2.2 — Sequential Chain (chaining two LLM calls)

You can chain two LLM calls together. The output of the first becomes input to the second.

In [16]:
# Chain 1: generate a topic idea
idea_prompt = ChatPromptTemplate.from_template(
    "Give me one creative blog post title about {subject}. "
    "Just the title, nothing else."
)

# Chain 2: write an outline for whatever title came out of Chain 1
outline_prompt = ChatPromptTemplate.from_template(
    "Write a 5-point outline for this blog post title: {title}"
)

# Use RunnablePassthrough to pass the original input AND the intermediate output
from langchain_core.runnables import RunnablePassthrough

full_chain = (
    {"title": idea_prompt | llm | parser,
     "subject": RunnablePassthrough()}   # pass the original input through
    | outline_prompt
    | llm
    | parser
)

result = full_chain.invoke({"subject": "artificial intelligence in education"})
print(result)


Here's a 5-point outline for the blog post "Teaching Machines to Teach: The Future of AI-Powered Personalized Learning":

**I. Introduction**

* Brief overview of the current state of education and the need for innovation
* Explanation of the concept of AI-powered personalized learning and its potential benefits
* Thesis statement: By harnessing the power of AI, we can create machines that teach, revolutionizing the way we learn and making education more accessible and effective.

**II. Understanding the Components of AI-Powered Personalized Learning**

* Discussion of the key components of AI-powered personalized learning, including:
	+ Natural Language Processing (NLP) for real-time feedback and assessment
	+ Machine Learning (ML) for adapting to individual learning styles and pace
	+ Data Analytics for tracking student progress and identifying areas of improvement
* Explanation of how these components work together to create a seamless and engaging learning experience.

**III. The B

### 📝 Task 2
> Build a 2-step chain that:
> 1. Asks the LLM to write a Python function for a task you describe
> 2. Then asks the LLM to add docstrings and comments to that function

> Hint: use `RunnablePassthrough` to pass the function code from step 1 into step 2.


In [20]:
# Task 2 — your sequential chain

step1_prompt = ChatPromptTemplate.from_template(
    "Write a Python function that {task}. Return only the code."
)

step2_prompt = ChatPromptTemplate.from_template(
    "Add a docstring and inline comments to this Python function:\n\n{code}"
)
# Use RunnablePassthrough to pass the original input AND the intermediate output
from langchain_core.runnables import RunnablePassthrough

my_chain = (
    {"code": step1_prompt | llm | parser,
     "task": RunnablePassthrough()}   # pass the original input through
    | step2_prompt
    | llm
    | parser
)

result = my_chain.invoke({"task": "sorts a list of dictionaries by a given key"})
print(result)
# Build your chain here
# my_chain = ...

# Test it
# print(my_chain.invoke({"task": "sorts a list of dictionaries by a given key"}))


Here's the function with a docstring and inline comments:

```python
def sort_dict_list(dict_list, key):
    """
    Sorts a list of dictionaries by a given key.

    Args:
        dict_list (list): A list of dictionaries.
        key (str): The key to sort by.

    Returns:
        list: The sorted list of dictionaries.
    """
    # Use the built-in sorted function to sort the list of dictionaries
    # The key function specifies the attribute to be used for sorting
    return sorted(
        # Pass the list of dictionaries to the sorted function
        dict_list,
        # Use a lambda function as the key to get the value associated with the given key
        # If the key is not found in a dictionary, return negative infinity to ensure it's sorted last
        key=lambda x: x.get(key, float('-inf'))
    )
```


---
## Part 3 — Output Parsers

AI output is text. Output parsers convert that text into **structured Python objects** (lists, dicts, Pydantic models) so your code can use the data programmatically.


### 3.1 — StrOutputParser (you already used this)


In [21]:
# StrOutputParser — just extracts the text string
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template("What are 3 uses of {technology}?")
    | llm
    | StrOutputParser()
)
print(chain.invoke({"technology": "machine learning"}))


Machine learning has numerous applications across various industries. Here are three common uses of machine learning:

1. **Image and Speech Recognition**: Machine learning is used in image recognition to identify objects, people, and patterns in images. For example, self-driving cars use machine learning algorithms to recognize road signs, pedestrians, and other vehicles. Similarly, speech recognition systems, such as virtual assistants like Siri and Alexa, use machine learning to recognize spoken words and phrases.

2. **Recommendation Systems**: Machine learning is used in recommendation systems to suggest products, movies, or music based on a user's preferences and behavior. For example, Netflix uses machine learning algorithms to suggest TV shows and movies based on a user's viewing history and ratings. Amazon also uses machine learning to recommend products based on a user's purchase history and browsing behavior.

3. **Predictive Maintenance**: Machine learning is used in predic

### 3.2 — JsonOutputParser — get a Python dict back


In [22]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_template(
    """Extract information from this job posting and return ONLY a JSON object.
No markdown, no explanation.

JSON keys required: role, seniority, skills (list), remote (bool)

Job posting: {posting}"""
)

json_chain = json_prompt | llm | JsonOutputParser()

result = json_chain.invoke({
    "posting": (
        "We are looking for a senior backend developer with Python and AWS experience. "
        "Remote-friendly position. 5+ years required."
    )
})

print(type(result))   # <-- dict, not a string!
print(result)
print("\nRole:", result.get("role"))
print("Remote:", result.get("remote"))


<class 'dict'>
{'role': 'senior backend developer', 'seniority': 'senior', 'skills': ['Python', 'AWS'], 'remote': True}

Role: senior backend developer
Remote: True


### 3.3 — Pydantic Output Parser — typed, validated data

The most robust option: define a schema with Pydantic and the parser validates every field.

In [23]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Define your expected output schema
class ProductReview(BaseModel):
    sentiment:  str   = Field(description="positive, neutral, or negative")
    score:      float = Field(description="score from 0.0 to 1.0")
    key_points: List[str] = Field(description="list of 3 key points from the review")

parser = PydanticOutputParser(pydantic_object=ProductReview)

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "You analyse product reviews. {format_instructions}"),
    ("human",  "Review: {review}")
])

chain = review_prompt | llm | parser

result = chain.invoke({
    "review": (
        "This laptop is incredible — the battery lasts 14 hours, "
        "the keyboard feels premium, but the webcam quality is poor."
    ),
    "format_instructions": parser.get_format_instructions()
})

print(type(result))          # ProductReview (Pydantic model)
print("Sentiment:", result.sentiment)
print("Score:",     result.score)
print("Points:",    result.key_points)


<class '__main__.ProductReview'>
Sentiment: positive
Score: 0.8
Points: ['the battery lasts 14 hours', 'the keyboard feels premium', 'the webcam quality is poor']


### 📝 Task 3
> Define a Pydantic model called `NewsArticle` with these fields:
> - `headline` (str)
> - `topic` (str)
> - `summary` (str — 1 sentence)
> - `is_breaking` (bool)

> Build a chain that takes any article text and returns a filled `NewsArticle` object.


In [27]:
# Task 3 — your Pydantic parser

from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class NewsArticle(BaseModel):
    headline:    str  = Field(description="The article headline")
    topic:       str  = Field(description="Main topic in one word")
    summary:     str  = Field(description="One sentence summary")
    is_breaking: bool = Field(description="Whether this is breaking news")

# Build your parser and chain here
# parser = ...
parser = PydanticOutputParser(pydantic_object=NewsArticle)

Article_prompt= ChatPromptTemplate.from_messages([
    ('system', 'you analyse news article.{format_instructions}'),
    ('human', 'Review:{sample_article}')
])
# chain  = ...
chain = Article_prompt| llm| parser

result=chain.invoke({
    'sample_article':(
        "Scientists at MIT announced today a major breakthrough in battery technology. "
        "The new solid-state battery charges in under 5 minutes and holds 3x more energy. "
        "This could revolutionise electric vehicles by end of 2026."
    ),
    'format_instructions': parser.get_format_instructions(),
})

print(type(result))
print("Headline:", result.headline)
print("Topic:",     result.topic)
print("Summary:",    result.summary)
print("Is_breaking:",    result.is_breaking)


#sample_article = (
 #   "Scientists at MIT announced today a major breakthrough in battery technology. "
  #  "The new solid-state battery charges in under 5 minutes and holds 3x more energy. "
   # "This could revolutionise electric vehicles by end of 2026."
#)

# result = chain.invoke({...})
# print(result)


<class '__main__.NewsArticle'>
Headline: MIT Scientists Unveil Breakthrough in Battery Technology
Topic: Battery
Summary: MIT scientists announce a major breakthrough in battery technology, charging in under 5 minutes and holding 3x more energy.
Is_breaking: True


---
## Part 4 — Conversation Memory

By default, each LLM call is stateless — the model forgets everything immediately.  
Memory lets you build chatbots that remember what was said.


### How memory works in LangChain

```
User says something
      ↓
Append to message history
      ↓
Send FULL history to LLM
      ↓
Append LLM reply to history
      ↓
Repeat
```


In [28]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# MessagesPlaceholder reserves a spot in the prompt for the conversation history
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly Python tutor for beginners."),
    MessagesPlaceholder(variable_name="history"),   # ← history goes here
    ("human", "{user_input}")
])

chain = chat_prompt | llm | StrOutputParser()

# We maintain the history manually as a list
history = []

def chat(user_input):
    response = chain.invoke({
        "user_input": user_input,
        "history":    history
    })
    # Add both turns to history so the model remembers them
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))
    return response

# Have a 3-turn conversation
print("Turn 1:", chat("What is a for loop in Python?"))
print()
print("Turn 2:", chat("Can you show me an example?"))
print()
print("Turn 3:", chat("What if I want to loop backwards?"))


Turn 1: A for loop in Python is a control structure that allows you to iterate over a sequence (such as a list, tuple, dictionary, or string) and execute a block of code for each item in the sequence.

The basic syntax of a for loop in Python is as follows:

```python
for variable in iterable:
    # code to be executed
```

Here's a breakdown of the components:

- `for`: This keyword is used to introduce the loop.
- `variable`: This is the variable that will take on the value of each item in the iterable on each iteration.
- `in`: This keyword is used to specify the iterable that the loop will iterate over.
- `iterable`: This is the sequence (such as a list, tuple, dictionary, or string) that the loop will iterate over.
- `# code to be executed`: This is the code that will be executed for each item in the iterable.

Here's an example of a for loop in Python:

```python
fruits = ['apple', 'banana', 'cherry']

for fruit in fruits:
    print(fruit)
```

In this example, the for loop will 

In [29]:
# Inspect the history — notice all messages are saved
print(f"History has {len(history)} messages:")
for msg in history:
    label = "USER" if isinstance(msg, HumanMessage) else "BOT"
    print(f"  [{label}] {msg.content[:80]}...")


History has 6 messages:
  [USER] What is a for loop in Python?...
  [BOT] A for loop in Python is a control structure that allows you to iterate over a se...
  [USER] Can you show me an example?...
  [BOT] Here are a few examples of using for loops in Python:

**Example 1: Printing a L...
  [USER] What if I want to loop backwards?...
  [BOT] To loop backwards in Python, you can use the `range` function with a negative st...


### 📝 Task 4
> Reset the history and build your own topic chatbot.  
> Pick a subject (cooking, travel, finance).  
> Have 4 turns where each question builds on the last answer.


In [30]:
# Task 4 — your own topic chatbot
my_history = []

my_prompt = ChatPromptTemplate.from_messages([
    ("system", "YOU ARE A FRIENDLY BIOLOGY TUTOR FOR BEGINNERS"),   # ← change this
    MessagesPlaceholder(variable_name="history"),
    ("human", "{user_input}")
])
my_chain = my_prompt | llm | StrOutputParser()

def my_chat(user_input):
    response = my_chain.invoke({
        "user_input": user_input,
        "history": my_history
    })

    my_history.append(HumanMessage(content=user_input))
    my_history.append(AIMessage(content=response))
    return response

# Turn 1
# print(my_chat("YOUR FIRST MESSAGE"))
print("Turn 1:", chat('What is a cell?'))
print()
print("Turn 2:", chat ("what are the types of cell?"))
print()
print("Turn 3:", chat("What are the functions of cell?"))

Turn 1: In Python, a cell is a single location in a data structure that stores a single value. It's a basic unit of storage that can hold a specific type of data, such as a number, a string, or a boolean value.

Think of a cell like a box that can hold a single item. You can put a book in one box, a toy in another, or a piece of paper in a third. Similarly, in a Python data structure like a list or an array, each cell holds a single value.

Here are a few examples of how cells work in Python:

**Example 1: A list of numbers**

```python
numbers = [1, 2, 3, 4, 5]
```

In this example, each element in the list (1, 2, 3, 4, and 5) is stored in a separate cell.

**Example 2: A string**

```python
greeting = "Hello, World!"
```

In this example, the entire string "Hello, World!" is stored in a single cell.

**Example 3: A dictionary**

```python
person = {'name': 'John', 'age': 30, 'city': 'New York'}
```

In this example, each key-value pair (name, John; age, 30; and city, New York) is sto

---
## Part 5 — Tools & Agents

A **tool** is a Python function the LLM can call.  
An **agent** decides *which* tools to use and *in what order* based on the task.


In [41]:

from langchain_core.tools import tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ── Define tools using the @tool decorator ────────────────────────────
@tool
def word_count(text: str) -> str:
    """Counts the number of words in a given text."""
    count = len(text.split())
    return f"The text contains {count} words."

@tool
def to_uppercase(text: str) -> str:
    """Converts text to uppercase."""
    return text.upper()

@tool
def calculate(expression: str) -> str:
    """Evaluates a simple math expression like '2 + 2' or '10 * 5'."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {e}"

tools = [word_count, to_uppercase, calculate]
print("Tools defined:", [t.name for t in tools])


Tools defined: ['word_count', 'to_uppercase', 'calculate']


In [42]:
# ── Create the agent ──────────────────────────────────────────────────
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools. "
               "Use tools when they help answer the question."),
    ("human",  "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")  # required for tool use
])

agent    = create_tool_calling_agent(llm, tools, agent_prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


In [43]:
# ── Let the agent decide which tools to use ───────────────────────────
result = executor.invoke({
    "input": "Count the words in this sentence: 'LangChain makes building LLM apps easy and fun.'"
})
print("\nFinal answer:", result["output"])




> Entering new AgentExecutor chain...

Invoking: `word_count` with `{'text': 'LangChain makes building LLM apps easy and fun.'}`


The text contains 8 words.
Invoking: `word_count` with `{'text': 'The text contains 8 words.'}`


The text contains 5 words.So the total number of words is 8 + 5 = 13.

> Finished chain.

Final answer: So the total number of words is 8 + 5 = 13.


In [44]:
# Agent decides to use the calculator tool
result = executor.invoke({
    "input": "What is 47 multiplied by 13?"
})
print("\nFinal answer:", result["output"])




> Entering new AgentExecutor chain...

Invoking: `calculate` with `{'expression': '47 * 13'}`


Result: 611I can provide more accurate results with the function available.

> Finished chain.

Final answer: I can provide more accurate results with the function available.


In [45]:
# Agent decides which tool(s) to call for a multi-part task
result = executor.invoke({
    "input": (
        "Count the words in 'Hello world this is a test' "
        "and also calculate 15 + 27."
    )
})
print("\nFinal answer:", result["output"])




> Entering new AgentExecutor chain...

Invoking: `word_count` with `{'text': 'Hello world this is a test'}`


The text contains 6 words.
Invoking: `calculate` with `{'expression': '15 + 27'}`


Result: 42The word count for the given text is 6 and the result of the calculation is 42.

> Finished chain.

Final answer: The word count for the given text is 6 and the result of the calculation is 42.


### 📝 Task 5
> Add a new tool called `reverse_text` that reverses any string.  
> Decorate it with `@tool`.  
> Rebuild the executor with this new tool added to the list.  
> Ask the agent to reverse a sentence **and** count its words.


In [46]:
# Task 5 — add a new tool

@tool
def reverse_text(text: str) -> str:
    """Reverses the characters in a given text."""
    return text[ ::-1]

    # TODO: return text reversed
    #pass

# Rebuild executor with reverse_text included
my_tools    = [word_count, to_uppercase, calculate, reverse_text]

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools. "
               "Use tools when they help answer the question."),
    ("human",  "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")  # required for tool use
])

my_agent    = create_tool_calling_agent(llm, my_tools, agent_prompt)
my_executor = AgentExecutor(agent=my_agent, tools=my_tools, verbose=True)

result = my_executor.invoke({
     "input": "Reverse 'LangChain is powerful' and count the words."
 })
print(result["output"])




> Entering new AgentExecutor chain...

Invoking: `reverse_text` with `{'text': 'LangChain is powerful'}`


lufrewop si niahCgnaL
Invoking: `word_count` with `{'text': 'lufrewop si niahCgnaL'}`


The text contains 3 words.The actual reversed text is 'powerful is LangChain' and the word count is 3.

> Finished chain.
The actual reversed text is 'powerful is LangChain' and the word count is 3.


---
## Part 6 — RAG (Retrieval-Augmented Generation)

RAG lets the LLM answer questions about **your own documents** — data it was never trained on.

### How RAG works

```
Your documents
      ↓
  Split into chunks
      ↓
  Embed each chunk → vector numbers
      ↓
  Store in a vector database (FAISS)
      ↓
User asks a question
      ↓
  Embed the question
      ↓
  Find the most similar chunks
      ↓
  Send: retrieved chunks + question → LLM
      ↓
  Answer grounded in YOUR data
```


In [49]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough

# ── Step 1: Our "documents" (in real life, these could be PDFs) ───────
docs_text = """
LangChain is an open-source framework for building LLM-powered applications.
It provides abstractions for prompts, chains, memory, tools, and agents.
LangChain supports many LLM providers including OpenAI, Anthropic, and Groq.

Groq is an AI hardware and software company with extremely fast inference speeds.
The Groq LPU (Language Processing Unit) can run Llama 3 at over 800 tokens per second.
Groq offers a free API tier that is excellent for learning and prototyping.

FAISS stands for Facebook AI Similarity Search.
It is a library for efficient similarity search over dense vectors.
LangChain integrates with FAISS to provide a simple vector store for RAG applications.
"""

# ── Step 2: Split into chunks ─────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks   = splitter.create_documents([docs_text])
print(f"Split into {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: {c.page_content[:60]}...")


Split into 6 chunks
  Chunk 0: LangChain is an open-source framework for building LLM-power...
  Chunk 1: LangChain supports many LLM providers including OpenAI, Anth...
  Chunk 2: Groq is an AI hardware and software company with extremely f...
  Chunk 3: Groq offers a free API tier that is excellent for learning a...
  Chunk 4: FAISS stands for Facebook AI Similarity Search.
It is a libr...
  Chunk 5: LangChain integrates with FAISS to provide a simple vector s...


In [50]:
# ── Step 3: Embed and store in FAISS ─────────────────────────────────
# HuggingFace embeddings run locally — no API key needed
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 2})   # top 2 chunks

print("✅ Vector store built with", vectorstore.index.ntotal, "vectors")


/tmp/ipykernel_10958/3577334244.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector store built with 6 vectors


In [51]:
# ── Step 4: Build the RAG chain ───────────────────────────────────────
rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Ask questions — the model can only answer from OUR documents
print(rag_chain.invoke("What is Groq?"))
print()
print(rag_chain.invoke("What does FAISS stand for?"))
print()
print(rag_chain.invoke("Who invented Python?"))   # not in our docs!


Groq is an AI hardware and software company with extremely fast inference speeds.

FAISS stands for Facebook AI Similarity Search.

I don't have that information.


### 📝 Task 6
> Replace `docs_text` with 3–5 paragraphs about a topic YOU know.  
> Rebuild the vector store and ask 3 questions.  
> Try one question that is NOT answered by your text — verify the model says it doesn't know.


In [52]:
# Task 6 — your own RAG
my_docs_text = """
Temperature controls the degree of randomness in token selection. Lower temperatures
are good for prompts that expect a more deterministic response, while higher temperatures
can lead to more diverse or unexpected results. A temperature of 0 (greedy decoding) is
deterministic: the highest probability token is always selected (though note that if two tokens
have the same highest predicted probability, depending on how tiebreaking is implemented
you may not always get the same output with temperature 0).

Temperatures close to the max tend to create more random output. And as temperature gets
higher and higher, all tokens become equally likely to be the next predicted token.

The Gemini temperature control can be understood in a similar way to the softmax function
used in machine learning. A low temperature setting mirrors a low softmax temperature (T),
emphasizing a single, preferred temperature with high certainty. A higher Gemini temperature
setting is like a high softmax temperature, making a wider range of temperatures around
the selected setting more acceptable. This increased uncertainty accommodates scenarios
where a rigid, precise temperature may not be essential like for example when experimenting
with creative output.
"""

# Split, embed, store, then query
my_splitter    = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
my_chunks      = my_splitter.create_documents([my_docs_text])
my_vectorstore = FAISS.from_documents(my_chunks, embeddings)   # reuse embeddings
my_retriever   = my_vectorstore.as_retriever(search_kwargs={"k": 2})

my_rag_chain = (
    {"context": my_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print(my_rag_chain.invoke("What is Temperature?"))
print(my_rag_chain.invoke("Explain the Gemini temperature?"))
print(my_rag_chain.invoke("List the function of the heart?"))


Temperature controls the degree of randomness in token selection. Lower temperatures are good for prompts that expect a more deterministic response, while higher temperatures are more suitable for open-ended or ambiguous prompts.
The Gemini temperature control is a system that emphasizes a single, preferred temperature with high certainty. It can be understood in a similar way to the softmax function used in machine learning, where a higher Gemini temperature setting is like a high softmax temperature, making a wider range of temperatures around it more likely.
I don't have that information.


---
## Part 7 — Capstone: Research Assistant 🏆

Combine everything into one system:

```
User question
      ↓
RAG retriever  →  relevant context
      ↓
Prompt Template  (system role + context + history + question)
      ↓
ChatGroq (llm)
      ↓
StrOutputParser → answer stored in history
      ↓
Loop
```


In [53]:
# ── Capstone: Conversational RAG assistant ────────────────────────────

knowledge = """
Python is a high-level, interpreted programming language created by Guido van Rossum in 1991.
It emphasises readability and uses significant indentation.
Python supports multiple programming paradigms: procedural, object-oriented, and functional.

Python is widely used in data science, machine learning, web development, and automation.
Popular libraries include NumPy, Pandas, Matplotlib for data science,
Django and Flask for web development, and TensorFlow and PyTorch for machine learning.

Python's package manager is pip. Virtual environments can be created with venv or conda.
The Python Package Index (PyPI) hosts over 400,000 packages.
Python 3 is the current major version; Python 2 reached end-of-life in 2020.
"""

# Build vector store
splitter_cap  = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks_cap    = splitter_cap.create_documents([knowledge])
vs_cap        = FAISS.from_documents(chunks_cap, embeddings)
retriever_cap = vs_cap.as_retriever(search_kwargs={"k": 2})

# Conversational prompt with history + retrieved context
conv_rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a knowledgeable Python tutor. "
     "Use the retrieved context to answer accurately. "
     "If the answer isn't in the context, say so honestly.\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

conv_history = []

def research_chat(question):
    # Retrieve relevant chunks for this question
    context_docs = retriever_cap.invoke(question)
    context_text = format_docs(context_docs)

    # Build and invoke the chain
    chain_cap = conv_rag_prompt | llm | StrOutputParser()
    answer = chain_cap.invoke({
        "question": question,
        "context":  context_text,
        "history":  conv_history
    })

    # Store in history
    conv_history.append(HumanMessage(content=question))
    conv_history.append(AIMessage(content=answer))
    return answer

print("Q1:", research_chat("When was Python created and who made it?"))
print()
print("Q2:", research_chat("What libraries did you just mention for data science?"))
print()
print("Q3:", research_chat("Which one should a beginner start with?"))


Q1: Python was created in 1991 by Guido van Rossum.

Q2: I mentioned the following libraries for data science:

1. NumPy
2. Pandas
3. Matplotlib

Q3: I would recommend starting with NumPy. NumPy is a foundational library for numerical computing in Python, and it provides an efficient way to work with arrays and mathematical operations. It's a fundamental library that many other libraries, including Pandas and Matplotlib, build upon.

Additionally, NumPy is relatively easy to learn and provides a lot of value even for simple tasks, making it a great starting point for beginners.


In [54]:
# ── Capstone: Conversational RAG assistant ────────────────────────────

knowledge = """
 The concept of Blood-Brain Barrier (BBB)– Based upon experiments of Paul Ehrlich (1885) & Edwin
Goldman (1909)
• Injection of water-soluble dyes into peripheral circulation → brain NOT stained;
CSF NOT colored, but heavy staining of the choroid plexus
• Injection of water-soluble dyes into subarachnoid space → brain stained;
peripheral tissues NOT stained; CSF colored
• Conversely, injection of lipid-soluble dyes into peripheral circulation → brain
stained; CSF colored
• Two barrier systems in the brain (postulated by Broman, 1941): – the Blood-Brain Barrier at the cerebral microvasculature– the Blood-CSF barrier at the choroid plex
"""

# Build vector store
splitter_cap  = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks_cap    = splitter_cap.create_documents([knowledge])
vs_cap        = FAISS.from_documents(chunks_cap, embeddings)
retriever_cap = vs_cap.as_retriever(search_kwargs={"k": 2})

# Conversational prompt with history + retrieved context
conv_rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a knowledgeable science tutor. "
     "Use the retrieved context to answer accurately. "
     "If the answer isn't in the context, say so honestly.\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

conv_history = []

def research_chat(question):
    # Retrieve relevant chunks for this question
    context_docs = retriever_cap.invoke(question)
    context_text = format_docs(context_docs)

    # Build and invoke the chain
    chain_cap = conv_rag_prompt | llm | StrOutputParser()
    answer = chain_cap.invoke({
        "question": question,
        "context":  context_text,
        "history":  conv_history
    })

    # Store in history
    conv_history.append(HumanMessage(content=question))
    conv_history.append(AIMessage(content=answer))
    return answer

print("Q1:", research_chat("Explain the blood brain barrie?"))
print()
print("Q2:", research_chat("The concept of the blood brain barrier was based whose experiments?"))
print()
print("Q3:", research_chat("Which year was the experiments done?"))


Q1: The Blood-Brain Barrier (BBB) is a highly selective permeable barrier that separates the brain's internal environment from the bloodstream. It is a crucial structure that plays a vital role in maintaining the brain's homeostasis and preventing the entry of toxins, pathogens, and other foreign substances into the brain.

The BBB is formed by the endothelial cells that line the brain's blood vessels, including capillaries, arterioles, and venules. These endothelial cells are tightly joined together by tight junctions, which prevent the free diffusion of molecules across the barrier.

There are several key features of the BBB:

1. **Selective permeability**: The BBB allows certain substances, such as oxygen, nutrients, and hormones, to pass through while preventing others, like toxins, pathogens, and large molecules.
2. **Endothelial cell junctions**: The tight junctions between endothelial cells form a seal that prevents the free diffusion of molecules across the barrier.
3. **Basal 

### 🎯 Your Turn
> Replace `knowledge` with text about YOUR chosen domain.  
> Run the capstone with 3 questions — make the 2nd and 3rd refer back to previous answers.

---

## ✅ What You've Learned

| Concept | What it does |
|---|---|
| `ChatPromptTemplate` | Reusable prompt with `{placeholders}` |
| LCEL `\|` pipe | Compose components into a pipeline |
| `StrOutputParser` | Extract plain text from model response |
| `JsonOutputParser` | Get a Python dict back |
| `PydanticOutputParser` | Get typed, validated data back |
| `MessagesPlaceholder` | Slot for conversation history in a prompt |
| `@tool` decorator | Turn any function into an LLM-callable tool |
| `AgentExecutor` | Run an agent that chooses its own tools |
| FAISS + embeddings | Store and search documents by meaning |
| RAG chain | Ground LLM answers in your own documents |

## Next Steps
- 📚 [LangChain Docs](https://python.langchain.com)
- 🔗 Add web search with `DuckDuckGoSearchRun` from `langchain_community.tools`
- 🗄️ Swap FAISS for a persistent DB like **Chroma** or **Pinecone**
- 🌐 Build a UI with **Streamlit**: `pip install streamlit`
